In [1]:
import gymnasium as gym

In [15]:
# the training environment
env = gym.make("CartPole-v1", render_mode="human")
# 'render_mode' specifies how the environment should be visualized
# 'human' -> a visual window
# 'rgb_array' -> image arrays
# None -> run without visuals

In [16]:
# reset env
observation, info = env.reset()

In [17]:
print(f"starting observation: {observation}")
# cart position, cart velocity, pole_angle, pole angular velocity

starting observation: [ 0.00953225  0.03355471  0.02575719 -0.04955927]


In [18]:
episode_over = False
total_reward = 0

In [19]:
while not episode_over:
  # choose an action
  action = env.action_space.sample() # random for now
  # take action
  observation, reward, terminated, truncated, info = env.step(action)

  # reward +1 for each step the pole stays upright
  # terminated: True if pole falls too far (agent failed)
  # truncated: True if we hit the time limit (500 steps)

  total_reward += reward
  episode_over = terminated or truncated

print(f"Episode finished! Total Reward: {total_reward}")
env.close()

Episode finished! Total Reward: 21.0


In [21]:
# using spaces
import gymnasium as gym
env = gym.make("CartPole-v1")
print(f"Action Space: {env.action_space}")
print(f"Sample Action: {env.action_space.sample()}")

# box observation space(continuous values)
print(f"Observation space: {env.observation_space}")
print(f"sample observation: {env.observation_space.sample()}")

Action Space: Discrete(2)
Sample Action: 1
Observation space: Box([-4.8               -inf -0.41887903        -inf], [4.8               inf 0.41887903        inf], (4,), float32)
sample observation: [ 3.5247364  -0.52480334  0.24544145  0.1488236 ]


In [26]:
## modifying an environment
!pip install gymnasium[box2d] -q

import gymnasium as gym
from gymnasium.wrappers import FlattenObservation

env = gym.make("CarRacing-v3")
print(env.observation_space.shape)

# wrap to flatten observation to a 1d array
wrapped_env= FlattenObservation(env)
print(wrapped_env.observation_space.shape)

# this allows you use algorithms that expect 1D input

(96, 96, 3)
(27648,)


### common wrappers
- TimeLimit: gives a truncated signal if a maximum number of timesteps has been exceeded

- ClipAction: Clips any action passed to step, ensures it is within the valid action space

- RescaleAction: Rescales actions to a dfieerenet range (useful for algorithms that output actions in [-1, 1] but environment expects [0, 10])

- TimeAwareObservation: Adds information about the current timestep to the observation

other wrappers -> https://gymnasium.farama.org/api/wrappers/

In [28]:
# to access a wrapped environment's original environment: use unwrapped
print(wrapped_env)
print(wrapped_env.unwrapped)

<FlattenObservation<TimeLimit<OrderEnforcing<PassiveEnvChecker<CarRacing<CarRacing-v3>>>>>>
<CarRacing<CarRacing-v3>>


## Training an Agent

In [58]:
from collections import defaultdict
import gymnasium as gym
import numpy as np

class BlackjackAgent:
  def __init__(self, env: gym.Env, learning_rate: float, init_epsilon: float, epsilon_decay: float, final_epsilon: float, discount_factor: float=0.95):
    """init a Q-learning agent.
    Args:
      env: the training environment
      learning_rate: how quick you update q vales
      init_epsilon: starting exploration rate (usually 1.0)
      epsilon_decay: how much to reduce epsilon per episode
      final_epsilon: lowest value of epsilon (usually 0.1)
      discount_factor: how much to value future rewards (0-1)
    """

    self.env = env
    # q table: maps (state, action) to expected reward
    # defaultdict creates entries with zeros for new states
    self.q_values = defaultdict(lambda: np.zeros(env.action_space.n))
    self.lr = learning_rate
    self.discount_factor = discount_factor


    # exploration params
    self.epsilon = init_epsilon
    self.epsilon_decay = epsilon_decay
    self.final_epsilon = final_epsilon

    # track learning progress
    self.training_error = []

  def get_action(self, obs: tuple[int, int, bool]) -> int:
    """Choose an action using epsioon-greedy strategy.

    Returns:
      action: 0 (stand) or 1 (hit)
    """
    # with e-greedy probability: explore(random action)
    if np.random.random() < self.epsilon:
      return self.env.action_space.sample() # random action, same as exploring?

    # with probability (1 - epsilon): exploit
    else:
      return int(np.argmax(self.q_values[obs]))

  def update(self, obs: tuple[int, int, bool], action:int, reward:float, terminated: bool, next_obs: tuple[int, int, bool]):
    """Update Q-values based on experience.
    This is the heart of Q-learning. learn from state, action, reward, next_state)
    """

    # what is the best we could do fromthe next state?
    # (zero if episode terminated - no future rewards possible)
    future_q_value = (not terminated) * np.max(self.q_values[next_obs])

    # what should the q value be? (bellman equation)
    target = reward + self.discount_factor * future_q_value

    # how wrong was our current estimate
    temporal_difference = target - self.q_values[obs][action]

    # update out estimate in the direction of the error
    # learning rate controls how big steps we take
    self.q_values[obs][action] = (
        self.q_values[obs][action] + self.lr * temporal_difference
    )

    # track learning
    self.training_error.append(temporal_difference)

  def decay_epsilon(self):
    """Reduce exploration rate after each episode"""
    self.epsilon = max(self.final_epsilon, self.epsilon - self.epsilon_decay)



In [59]:
env = gym.make("Blackjack-v1")
q_values = defaultdict(lambda: np.zeros(env.action_space.n))
q_values

defaultdict(<function __main__.<lambda>()>, {})